# DataPilot AI — Exploratory Data Analysis (EDA)

**Purpose:** Inspect Dataset A after collection and preprocessing, and review chunking outcomes.

This notebook supports academic traceability for the Master's project. It explains **what** we measure, **why**, and **what the numbers mean**.

> Run from the repository root (or ensure the project root is on `sys.path`).\n> Prefer re-running the scripts if data changes:\n> - `python scripts/collect_documents.py`\n> - `python scripts/preprocess_documents.py`\n> - `python scripts/build_chunks.py`

## 1. Setup

We load JSON artefacts produced by the reproducible CLI pipelines rather than scraping again inside the notebook.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = "/content/drive/MyDrive/Masters_Project"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT_DIR)
except ImportError:
    pass

ROOT = Path.cwd()
if not (ROOT / "config").exists() and (ROOT.parent / "config").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pre_stats_path = ROOT / "data" / "processed" / "stats" / "preprocessing_stats.json"
chunk_stats_path = ROOT / "data" / "processed" / "stats" / "chunking_stats.json"
docs_jsonl = ROOT / "data" / "processed" / "documents.jsonl"
rejected_jsonl = ROOT / "data" / "processed" / "rejected.jsonl"
chunks_jsonl = ROOT / "knowledge_base" / "chunks" / "chunks.jsonl"

print("ROOT:", ROOT)
print("pre_stats exists:", pre_stats_path.exists())
print("chunk_stats exists:", chunk_stats_path.exists())

## 2. Preprocessing summary

**What:** Counts of accepted vs rejected documents, characters/tokens, duplicates.\n**Why:** Demonstrates data quality and cleanliness for academic assessment.\n**Meaning:** High acceptance with only intentional duplicate removals indicates a usable RAG corpus; empty/malformed counts should stay near zero.

In [ ]:
pre_stats = json.loads(pre_stats_path.read_text(encoding="utf-8"))
display_keys = [
    "total_input_documents",
    "accepted_documents",
    "rejected_documents",
    "total_sources",
    "total_characters",
    "total_approx_tokens",
    "duplicate_count",
    "empty_document_count",
    "malformed_document_count",
    "final_chunk_count",
]
for k in display_keys:
    print(f"{k}: {pre_stats.get(k)}")
print("\nDocuments per source:", pre_stats.get("documents_per_source"))
print("Documents per category:", pre_stats.get("documents_per_category"))

## 3. Document-level distributions

Plot accepted documents by source/category and inspect document length distribution.\nLong documents are expected for reference manuals (e.g., PostgreSQL SELECT); they will be chunked next.

In [ ]:
docs = [json.loads(line) for line in docs_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]
df = pd.DataFrame(docs)
print(df[["document_id", "source_id", "category", "char_count", "approx_token_count"]].head())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df["source_id"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#2F5D50")
axes[0].set_title("Docs per source")
df["category"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#3D7A6A")
axes[1].set_title("Docs per category")
df["char_count"].plot(kind="hist", bins=15, ax=axes[2], color="#6B8F71")
axes[2].set_title("Document char_count")
plt.tight_layout()
plt.show()

print(df["char_count"].describe())

## 4. Rejections

Rejected rows should mostly be **exact content duplicates** caused by curated topics that intentionally share one official page (documented in inventory notes). This is desirable for RAG so the same text is not indexed multiple times.

In [ ]:
if rejected_jsonl.exists() and rejected_jsonl.stat().st_size > 0:
    rejected = [json.loads(line) for line in rejected_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]
    rdf = pd.DataFrame(rejected)
    print("Rejected:", len(rdf))
    cols = [c for c in ["document_id", "source_id", "is_duplicate", "duplicate_of", "rejection_reasons"] if c in rdf.columns]
    display(rdf[cols])
else:
    print("No rejected documents.")

## 5. Chunking overview

**What:** Final chunk count and chunks per category/source.\n**Why:** Chunk size/overlap strongly affect retrieval quality and context window usage.\n**Meaning:** Starting config uses ~650 token chunks with ~75 token overlap (`config/rag.yaml`). These are **not** claimed optimal; later experiments can compare alternatives.

In [ ]:
if not chunk_stats_path.exists():
    print("Chunk stats missing. Run: python scripts/build_chunks.py")
else:
    chunk_stats = json.loads(chunk_stats_path.read_text(encoding="utf-8"))
    for k in [
        "documents_chunked",
        "final_chunk_count",
        "chunk_size_tokens",
        "chunk_overlap_tokens",
        "chunks_per_source",
        "chunks_per_category",
        "approx_token_summary",
    ]:
        print(f"{k}: {chunk_stats.get(k)}")

In [ ]:
if chunks_jsonl.exists():
    chunks = [json.loads(line) for line in chunks_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]
    cdf = pd.DataFrame(chunks)
    print("Chunks:", len(cdf))
    print(cdf[["chunk_id", "source_id", "category", "approx_token_count", "url"]].head())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    cdf["source_id"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#2F5D50")
    axes[0].set_title("Chunks per source")
    cdf["approx_token_count"].plot(kind="hist", bins=20, ax=axes[1], color="#6B8F71")
    axes[1].set_title("Chunk approx token counts")
    plt.tight_layout()
    plt.show()

    # Provenance sanity check
    missing = cdf["url"].isna().sum() + (cdf["url"] == "").sum()
    print("Chunks missing URL:", int(missing))
else:
    print("chunks.jsonl not found")

## 6. Interpretation for the research paper

- Dataset A is curated official documentation with preserved provenance (URL, source, category, topic).
- Preprocessing removed boilerplate and exact duplicates, improving corpus cleanliness.
- Chunking produces retrieval units with metadata required for grounded answers and citation.
- Embeddings + FAISS, RAG, LoRA, and LLM A/B/C tables are already executed (see notebooks 02, 04, 05, 06).

Do **not** invent metrics beyond the artefacts loaded above.